#Time Travel y Optimización basico en Delta Lake

#### Primero creamos la tabla de ejemplo donde aplicamos los conceptos

Luego realizamos una serie de operaciones básicas para generar distintas versiones de la tabla y poder trabajar con Time Travel:

- Insertamos un primer conjunto de datos en la tabla.
- Realizamos un segundo insert agregando nuevos registros.
- Ejecutamos un `UPDATE` para modificar información existente.
- Eliminamos una fila específica utilizando un `DELETE`.

Estas operaciones nos permiten generar múltiples versiones de la tabla en Delta Lake, lo cual es clave para explorar funcionalidades como Time Travel y optimización.


In [0]:
%sql
create table workspace.prueba.table_users (
  id int,
  name string,
  email string,
  age int,
  created timestamp
  ) using delta;

In [0]:
%sql
insert into workspace.prueba.table_users values (
  1, "fabrizio", "fabriziolanzettidev@gmail.com", 26, current_timestamp
)

In [0]:
%sql
insert into workspace.prueba.table_users
values
  (2, "maria", "maria@gmail.com", 25, current_timestamp),
  (3, "pablo", "pablo@gmail.com", 24, current_timestamp),
  (4, "jose", "jose@gmail.com", 23, current_timestamp)


In [0]:
%sql
update workspace.prueba.table_users
set email = "pablo_perez@gmail.com"
where id = "3"

In [0]:
%sql

delete from workspace.prueba.table_users
where id = "4"

####Podemos acceder a distintos estados de la tabla gracias a Delta Lake:

- Estado actual: consultando la tabla normalmente, obtenemos la última versión disponible.
- Por versión: podemos viajar a una versión específica de la tabla utilizando su número de versión.
- Por fecha y hora: también es posible consultar la tabla en un momento exacto en el tiempo.

Cada operación realizada (INSERT, UPDATE, DELETE, OPTIMIZE, etc.) genera un nuevo snapshot consistente de la tabla, lo que garantiza integridad y permite reproducir el estado exacto de los datos en cualquier punto del historial.


> Databricks puede ejecutar `OPTIMIZE` automáticamente (Auto Optimize) para compactar archivos y mejorar el rendimiento.

Auto Optimize incluye dos partes: **Optimized Writes**, que reduce la cantidad de archivos generados al escribir, y **Auto Compaction**, que ejecuta una versión liviana de `OPTIMIZE` después de cada escritura agrupando archivos pequeños.

Esto se puede ver en `DESCRIBE HISTORY`, donde aparece la operación `OPTIMIZE` con `"auto": "true"`, generando nuevas versiones en el historial de Delta Lake.


In [0]:
%sql
describe history workspace.prueba.table_users

Ahora aplicamos Time Travel para volver al estado de la tabla en el punto que deseemos.

- Utilizamos `VERSION AS OF` para consultar una versión específica de la tabla.
- Utilizamos `TIMESTAMP AS OF` para consultar la tabla en un momento exacto en el tiempo.

Esto nos permite recuperar estados anteriores de manera sencilla y confiable gracias al historial de Delta Lake.


In [0]:
%sql
--select * from workspace.prueba.table_users timestamp as of "2026-04-27 18:36:31" - Para volver a una version anterior con fechas
select * from workspace.prueba.table_users version as of 1 -- Para volver a una version anterior con versiones


Hasta ahora solo exploramos versiones anteriores. Si ejecutamos:

`SELECT * FROM workspace.prueba.tabe_users
`
obtendremos el estado actual de la tabla (sin el registro id 4 y con el id 3 actualizado).

Para volver realmente a un estado anterior, debemos utilizar:

`RESTORE TABLE`
Esto restaura la tabla a una versión previa, modificando su estado actual.


In [0]:
%sql
select * from workspace.prueba.table_users

In [0]:
%sql
restore table workspace.prueba.table_users to version as of 2

In [0]:
%sql
select * from workspace.prueba.table_users

Listo ya tenemos la tabla restaurada con el registro id 4 restaurado y con el registro id 3 con el email sin actualizar
- El RESTORE tambien va a quedar guardado en `DESCRIBE HISTORY`

In [0]:
%sql
optimize workspace.prueba.table_users zorder by (id)

## Comando OPTIMIZE y ZORDER
> El comando `OPTIMIZE` permite mejorar el rendimiento compactando archivos pequeños en archivos más grandes.

Además, se puede utilizar **ZORDER** para reorganizar los datos en función de columnas específicas, lo que optimiza la búsqueda y acelera las consultas filtradas por esas columnas.

#### Utilizar ZORDER en:
- Columnas de alta cardinalidad (maximo 3 columas)
- Columnas frecuentes en filtros
- Columnas justas usadas en queries

Esta operación también genera una nueva versión de la tabla dentro del historial de Delta Lake.


